<a href="https://colab.research.google.com/github/Pshyam17/ML_Project_Torch/blob/main/Mamba_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Understanding Mamba and State-Space Models (SSMs)

Mamba is a novel deep learning architecture that combines the strengths of both recurrent neural networks (RNNs) and convolutional neural networks (CNNs). It achieves linear-time complexity with respect to sequence length, enabling efficient processing of very long sequences, while maintaining high performance comparable to Transformers.

At its core, Mamba relies on a **State-Space Model (SSM)**. SSMs are a powerful class of models used to describe dynamic systems, representing them through a hidden 'state' that evolves over time. They are commonly used in control theory and signal processing.

### Continuous-Time State-Space Models

In their continuous form, a linear time-invariant (LTI) State-Space Model is defined by a set of differential equations:

1.  **State Equation**: `h'(t) = A h(t) + B x(t)`
2.  **Output Equation**: `y(t) = C h(t) + D x(t)`

Where:
*   `x(t)`: Input signal at time `t`
*   `h(t)`: Hidden state vector at time `t`
*   `y(t)`: Output signal at time `t`
*   `A`: State matrix (determines how the state evolves internally)
*   `B`: Input matrix (determines how the input affects the state)
*   `C`: Output matrix (determines how the state is mapped to the output)
*   `D`: Feedthrough matrix (determines how the input directly affects the output, often zero in neural network contexts)

These matrices (`A`, `B`, `C`, `D`) are constants for LTI systems. The hidden state `h(t)` encapsulates all necessary information from past inputs to predict future outputs.

### Discretization of State-Space Models (Zero-Order Hold - ZOH)

Neural networks operate on discrete sequences, so we need to convert the continuous SSM into a discrete-time equivalent. The **Zero-Order Hold (ZOH)** method is a common way to do this. It assumes that the input `x(t)` remains constant over a small time interval `Δ` (the sampling period).

Given a time step `Δ`, the continuous matrices `A` and `B` are transformed into discrete matrices `A_bar` and `B_bar`:

1.  **Discrete State Matrix (`A_bar`)**:
    `A_bar = exp(A * Δ)`
    Where `exp` is the matrix exponential.

2.  **Discrete Input Matrix (`B_bar`)**:
    `B_bar = (A * Δ)^-1 (exp(A * Δ) - I) B` (if `A` is invertible, where `I` is the identity matrix)
    A more general form using integration is:
    `B_bar = ∫_0^Δ exp(Aτ) B dτ`

With `A_bar` and `B_bar`, the discrete-time SSM equations become:

1.  **Discrete State Recurrence**: `h_k = A_bar h_{k-1} + B_bar x_k`
2.  **Discrete Output Equation**: `y_k = C h_k + D x_k`

This recurrence relation allows us to compute the state and output at each discrete time step `k` based on the previous state and the current input. This is the core mechanism that enables RNN-like behavior in SSMs.

In [4]:
import torch

def discretize_ssm_zoh(A, B, delta):
    
    #Discretizes continuous-time SSM matrices A and B using Zero-Order Hold.
    """
    A: (N, N) tensor
    B: (N, 1) tensor or (N, D_in)
    delta: scalar tensor
    returns: A_bar, B_bar
    """

    N = A.shape[0]
    D_in = B.shape[1] # infer D_in from shape of B

    # construct augmented matrix F_aug
    # F_aug = [[A*delta, B*delta],
    #          [0 (D_in, N), 0 (D_in, D_in)]]

    # top block [A*delta, B*delta]
    top_block = torch.cat([A * delta, B * delta], dim=1) # Shape (N, N + D_in)

    # bottom block: zeros of shape (D_in, N + D_in)
    bottom_block = torch.zeros(D_in, N + D_in, device=A.device)

    # combine them to form F_aug
    F = torch.cat([top_block, bottom_block], dim=0) # shape is (N + D_in, N + D_in)

    F_exp = torch.matrix_exp(F)

    # extract A_bar and B_bar_term from the exp augmented matrix
    A_bar = F_exp[:N, :N]
    B_bar_term = F_exp[:N, N:]

    return A_bar, B_bar_term

def discrete_ssm_step(h_prev, x_curr, A_bar, B_bar, C, D):
    """
     one step of a discrete-time SSM
    h_prev is (N,) tensor (previous hidden state)
    x_curr is (D_in,) tensor (current input)
    A_bar is (N, N) tensor
    B_bar is (N, D_in) tensor
    C is (D_out, N) tensor
    D is (D_out, D_in) tensor
    this returns : h_curr, y_curr
    """

    # making sure x_curr has a batch dimension or is compatible for matmul
    # if B_bar is (N, D_in) and x_curr is (D_in,), B_bar @ x_curr will be (N,)
    # if B_bar is (N, D_in) and x_curr is (batch_size, D_in), then (B_bar @ x_curr.T).T will be (batch_size, N)
    # assume single input vector x_curr

    # State update -->  h_k = A_bar h_{k-1} + B_bar x_k
    # (N,N) @ (N,) + (N, D_in) @ (D_in,)
    h_curr = torch.matmul(A_bar, h_prev) + torch.matmul(B_bar, x_curr)

    # Output update -->  y_k = C h_k + D x_k
    # (D_out, N) @ (N,) + (D_out, D_in) @ (D_in,)
    y_curr = torch.matmul(C, h_curr) + torch.matmul(D, x_curr)

    return h_curr, y_curr


ModuleNotFoundError: No module named 'torch'

In [ ]:


# define dimensions
N = 4  # state 
D_in = 2 # input 
D_out = 1 # output 

#  continuous-time SSM matrices (random init)
A_cont = torch.randn(N, N)
B_cont = torch.randn(N, D_in)
C_cont = torch.randn(D_out, N)
D_cont = torch.randn(D_out, D_in)

# time step
delta = torch.tensor(0.1)

# discretize the SSM matrices
A_bar, B_bar = discretize_ssm_zoh(A_cont, B_cont, delta)

print(f"cont A shape: {A_cont.shape}")
print(f"cont B shape: {B_cont.shape}")
print(f"discrete A_bar shape: {A_bar.shape}")
print(f"discrete B_bar shape: {B_bar.shape}")

# init hidden state and input
h_0 = torch.zeros(N) # init state
x_1 = torch.randn(D_in) # first input

print(f"\n the initial hidden state h_0: {h_0}")
print(f"first input x_1 is {x_1}")

#  one step of discrete SSM
h_1, y_1 = discrete_ssm_step(h_0, x_1, A_bar, B_bar, C_cont, D_cont)

print(f"\nUpdated hidden state h_1: {h_1}")
print(f"Output y_1: {y_1}")

# sim a sequence (5 steps)
sequence_length = 5
inputs = torch.randn(sequence_length, D_in)
hidden_states = [h_0]
outputs = []

current_h = h_0
for i in range(sequence_length):
    x_k = inputs[i]
    current_h, y_k = discrete_ssm_step(current_h, x_k, A_bar, B_bar, C_cont, D_cont)
    hidden_states.append(current_h)
    outputs.append(y_k)

print(f"\nSimulated sequence over {sequence_length} steps:")
for i in range(sequence_length):
    print(f"Step {i+1}: Input={inputs[i]}, Output={outputs[i]}")


Continuous A shape: torch.Size([4, 4])
Continuous B shape: torch.Size([4, 2])
Discrete A_bar shape: torch.Size([4, 4])
Discrete B_bar shape: torch.Size([4, 2])

Initial hidden state h_0: tensor([0., 0., 0., 0.])
First input x_1: tensor([ 0.5753, -0.9694])

Updated hidden state h_1: tensor([ 0.0512, -0.1275,  0.2890,  0.0759])
Output y_1: tensor([-0.5377])

Simulated sequence over 5 steps:
Step 1: Input=tensor([-1.0341,  1.3703]), Output=tensor([0.8811])
Step 2: Input=tensor([-1.1473,  0.4054]), Output=tensor([0.9988])
Step 3: Input=tensor([0.3368, 0.5628]), Output=tensor([0.3623])
Step 4: Input=tensor([-0.4131,  0.3567]), Output=tensor([0.7748])
Step 5: Input=tensor([ 0.4198, -0.6904]), Output=tensor([0.1505])


## The Selective State-Space Model (S-SSM)

The main innovation in Mamba, derived from the Selective State Space (S4) models, is the concept of **selectivity**. 

This means that the parameters of the SSM, specifically the discretization step `Δ`, and the input and output matrices `B` and `C`  are no longer FIXED constants but are dynamically computed based on the input!

 This dynamic adaptation allows the model to filter relevant information from the input sequence and ignore irrelevant parts, providing a powerful content-aware mechanism

In Mamba, for each input token `x_k` in a sequence, we calculate a new `Δ_k`, `B_k`, and `C_k`:

*   `Δ_k = f_Δ(x_k)`: A scalar or vector representing the time-step, usually generated by a small neural network (`f_Δ`) from the input token.
*   `B_k = f_B(x_k)`: The input matrix, generated by another network (`f_B`).
*   `C_k = f_C(x_k)`: The output matrix, generated by another network (`f_C`).

The `A` matrix, often stays fixed, learned during training. The `D` matrix is typically constant or omitted.

with these input-dependent parameters, the discrete-time SSM equations become:

1.  **Discrete State Recurrence**: `h_k = A_bar(Δ_k) h_{k-1} + B_bar(Δ_k, B_k) x_k`
2.  **Discrete Output Equation**: `y_k = C_k h_k + D x_k`

Where `A_bar(Δ_k)` and `B_bar(Δ_k, B_k)` are derived from the continuous `A` and `B_k` using the input-dependent `Δ_k`. This makes the state update process content-aware, allowing for efficient and effective long-range dependency modeling.

### Adapting Discretization for Selectivity

To implement the Selective Scan, we need to modify our `discretize_ssm_zoh` function to accept an input-dependent `B` and `Δ`. For Mamba, typically `Δ` is a scalar (or a vector of scalars for different state dimensions) and `B` is a vector derived from the input. For simplicity, we will adapt our existing `discretize_ssm_zoh` function to take a `delta_k` and `B_k` that can vary per step.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


def discretize_ssm_zoh_selective(A, B_k, delta_k):
    # A: (N,) diagonal elements, all negative
    # B_k: (N, D_in)
    # delta_k: scalar
    # for diagonal A the zoh formula simplifies to closed-form element-wise ops:
    #   A_bar = exp(A * delta)   <- no matrix exp needed
    #   B_bar = (A_bar - 1) / A * B  <- divide then broadcast over D_in
    # tried augmented matrix_exp first - correct but O(N^3) and wasteful when A is diagonal
    A_bar = torch.exp(A * delta_k)                        # (N,)
    B_bar = ((A_bar - 1) / A).unsqueeze(-1) * B_k         # (N, D_in)
    return A_bar, B_bar

def discrete_ssm_step_selective(h_prev, x_curr, A_bar, B_bar, C_k, D):
    # h_prev: (N,)
    # x_curr: (D_in,)
    # A_bar: (N,) diagonal --> element-wise multiply, not matmul
    # B_bar: (N, D_in)
    # C_k: (D_out, N)
    # D: (D_out, D_in)
    h_curr = A_bar * h_prev + B_bar @ x_curr   # (N,)
    y_curr = C_k @ h_curr + D @ x_curr         # (D_out,)
    return h_curr, y_curr


### 1. Data Preparation: Character-level Language Modeling

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

#small, classic text dataset

text = """
'Tis but thy name that is my enemy;
Thou art thyself, though not a Montague.
What's Montague? it is nor hand, nor foot,
Nor arm, nor face, nor any other part
Belonging to a man. O, be some other name!
What's in a name? that which we call a rose
By any other word would smell as sweet;
So Romeo would, were he not Romeo call'd,
Retain that dear perfection which he owes
Without that title. Romeo, doff thy name;
And for that name which is no part of thee
Take all myself.
"""

# creating vocabulary and mappings
chars = sorted(list(set(text)))
vocab_size = len(chars)
char_to_int = {ch: i for i, ch in enumerate(chars)}
int_to_char = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [char_to_int[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([int_to_char[i] for i in l]) # decoder: take a list of integers, output a string

print(f"original text length: {len(text)} characters")
print(f"vocabulary size: {vocab_size} unique characters")
print(f"First 10 encoded characters: {encode(text[:10])}")
print(f"decoded back: {decode(encode(text[:10]))}")

# convert the entire text to a PyTorch tensor
data = torch.tensor(encode(text), dtype=torch.long)
print(f"\nEncoded data tensor shape: {data.shape}")


Original text length: 484 characters
Vocabulary size: 38 unique characters
First 10 encoded characters: [0, 3, 15, 25, 33, 1, 18, 35, 34, 1]
Decoded back: 
'Tis but 

Encoded data tensor shape: torch.Size([484])


In [ ]:
# split data into training and validation sets
train_ratio = 0.9
n = int(train_ratio * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Training data size: {len(train_data)}")
print(f"Validation data size: {len(val_data)}")

# Function to get batches for character-level prediction
def get_batch(split, block_size, batch_size):
    data = train_data if split == 'train' else val_data
    # Generate random starting indices for the sequences in the batch
    ix = torch.randint(len(data) - block_size, (batch_size,))
    # Stack sequences of length block_size as inputs
    x = torch.stack([data[i:i+block_size] for i in ix])
    # Stack next characters as targets
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

block_size = 8 # Context length for predictions
batch_size = 4 # How many independent sequences will we process in parallel?

x, y = get_batch('train', block_size, batch_size)

print(f"\nInput batch (x) shape: {x.shape}") # (batch_size, block_size)
print(f"Target batch (y) shape: {y.shape}") # (batch_size, block_size)

print("\nSample input sequence (x[0]):")
print(decode(x[0].tolist()))
print("Sample target sequence (y[0]):")
print(decode(y[0].tolist()))


Training data size: 435
Validation data size: 49

Input batch (x) shape: torch.Size([4, 8])
Target batch (y) shape: torch.Size([4, 8])

Sample input sequence (x[0]):
thyself,
Sample target sequence (y[0]):
hyself, 


### 2. MambaBlock: Selective State-Space Layer

`MambaBlock` is the core computational unit of Mamba.  Each token's step size
Δ, input matrix B, and output matrix C are projected from the input itself; this is the selectivity that lets the model decide what to remember or forget
at each position.

Key design choices:
- **Diagonal A in log-space**: `A = -exp(A_log)` keeps eigenvalues strictly
  negative, ensuring the state decays rather than explodes.  Geometric init
  gives each state dimension a different natural timescale.
- **Two-stage delta projection**: `d_inner → dt_rank → d_inner` with a
  bias-corrected softplus keeps Δ inside `[dt_min, dt_max]`.
- **Depthwise conv1d** (kernel 4): cheap local context mixing before the scan,
  no cross-channel interaction.
- **Gated output**: the z-branch passes through SiLU and multiplies the SSM
  output, acting as a content-dependent gate.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# reuse discretize_ssm_zoh_selective and discrete_ssm_step_selective from cell 9

class MambaBlock(nn.Module):
    def __init__(self, d_model, N, dt_min=0.001, dt_max=0.1, expand=2):
        super().__init__()
        self.d_model = d_model
        self.N = N
        self.dt_min = dt_min
        self.dt_max = dt_max
        self.expand = expand
        self.d_inner = d_model * expand

        # diagonal A stored in log-space - enforces A = -exp(A_log) < 0 always
        # init: A_log[n] = log(n+1), so A[n] = -(n+1)
        # s4d-style geometric spread across state dims:
        #   - small n -> shorter timescale (fast decay)
        #   - large n -> longer timescale (slow decay)
        # tried full (N,N) randn before - gradient updates blew up, eigenvalues went positive
        A_log_init = torch.tensor([math.log(n + 1) for n in range(N)], dtype=torch.float32)
        self.A_log = nn.Parameter(A_log_init)

        # per-state bias broadcast over d_inner channels prevents hidden-state collapse
        self.b_c = nn.Parameter(torch.zeros(self.N))

        # expand + gate: one projection to get both branches
        # x_branch goes through the ssm, z_branch is the silu gate
        self.in_proj = nn.Linear(d_model, self.d_inner * 2, bias=False)
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

        # two-stage delta projection: d_inner -> dt_rank -> d_inner
        # low-rank bottleneck cuts params while giving each of d_inner channels its own delta
        # tried Linear(d_model, 1) before - single scalar delta can't adapt per-channel
        self.dt_rank = max(1, d_model // 16)
        self.delta_proj = nn.Linear(self.d_inner, self.dt_rank, bias=False)
        self.delta_proj_up = nn.Linear(self.dt_rank, self.d_inner, bias=True)
        # bias init: softplus(log(expm1(1))) = 1.0 at init, lands in [dt_min, dt_max] after clamp
        nn.init.constant_(self.delta_proj_up.bias, math.log(math.expm1(1)))

        # B and C depend on x_branch that's the selectivity point
        # per-token projections, shared across d_inner channels
        self.B_proj = nn.Linear(self.d_inner, N, bias=False)
        self.C_proj = nn.Linear(self.d_inner, N, bias=False)

        # depthwise conv for local context before the ssm scan
        # kernel_size=4 matches the paper - each token sees 3 positions back
        # groups=d_inner makes it depthwise: no cross-channel mixing, just temporal smoothing
        # padding=3 + slicing in forward keeps seq_len intact
        self.conv1d = nn.Conv1d(
            self.d_inner, self.d_inner, kernel_size=4,
            groups=self.d_inner, padding=3, bias=True
        )

    def forward(self, x_seq):
        # x_seq: (seq_len, d_model)
        seq_len = x_seq.shape[0]

        # project to 2*d_inner and split into ssm input and gate
        xz = self.in_proj(x_seq)                        # (seq_len, d_inner * 2)
        x_branch, z_branch = xz.chunk(2, dim=-1)        # both (seq_len, d_inner)

        # depthwise conv over x_branch before the scan adds local context
        # conv1d expects (batch, channels, seq_len), so reshape, apply, restore
        x_branch = x_branch.T.unsqueeze(0)                        # (1, d_inner, seq_len)
        x_branch = F.silu(self.conv1d(x_branch)[..., :seq_len])   # trim padding, apply silu
        x_branch = x_branch.squeeze(0).T                          # (seq_len, d_inner)

        # d_inner parallel ssms, each with N state dims
        h = torch.zeros(self.d_inner, self.N, device=x_seq.device)
        A_diag = -torch.exp(self.A_log)                  # (N,) always negative

        ssm_outputs = []
        for i in range(seq_len):
            x_k = x_branch[i]                            # (d_inner,)

            # delta pipeline: x_branch -> low-rank -> softplus -> clamp
            delta = F.softplus(self.delta_proj_up(self.delta_proj(x_k)))  # (d_inner,)
            delta = delta.clamp(self.dt_min, self.dt_max)

            # B and C depend on x_branch per-token, shared across channels
            B_k = self.B_proj(x_k)                       # (N,)
            C_k = self.C_proj(x_k)                       # (N,)

            # discretize: A_bar[c, n] = exp(A[n] * delta[c])
            # outer product of A_diag (N,) and delta (d_inner,) -> (d_inner, N)
            A_bar = torch.exp(A_diag.unsqueeze(0) * delta.unsqueeze(-1))   # (d_inner, N)
            B_bar = ((A_bar - 1) / A_diag.unsqueeze(0)) * B_k.unsqueeze(0) # (d_inner, N)

            # state update: each channel driven by its own scalar x_k[c]
            h = A_bar * h + B_bar * x_k.unsqueeze(-1)    # (d_inner, N)
            h = h + self.b_c                              # per-state bias, broadcast over d_inner

            # output: C_k collapses N state dims to a scalar per channel
            y_k = h @ C_k                                 # (d_inner,)
            ssm_outputs.append(y_k)

        ssm_out = torch.stack(ssm_outputs, dim=0)        # (seq_len, d_inner)

        # silu gate: z_branch controls what ssm output passes through
        output = ssm_out * F.silu(z_branch)              # (seq_len, d_inner)

        return self.out_proj(output)                      # (seq_len, d_model)


In [ ]:

d_model = 128
N = 16
seq_len = 10

input_sequence = torch.randn(seq_len, d_model)
mamba_block = MambaBlock(d_model=d_model, N=N)
output_sequence = mamba_block(input_sequence)

print(f'input:  {input_sequence.shape}')
print(f'output: {output_sequence.shape}')  # should be (seq_len, d_model)
print(output_sequence[:3])


Input sequence shape: torch.Size([10, 128])
Output sequence shape: torch.Size([10, 1])
First 3 outputs:
tensor([[  5.9840],
        [-20.6541],
        [ 62.7701]], grad_fn=<SliceBackward0>)


### 3. Full Mamba Model Stack

Three components wrap `MambaBlock` into a complete language model:

- **RMSNorm**: root-mean-square normalisation without mean subtraction.
  Standard LayerNorm hurt training on short sequences; RMSNorm was more stable.
- **ResidualBlock**: pre-norm residual wrap around `MambaBlock`.  Pre-norm
  (normalise before the layer) is more stable than post-norm at depth.
- **Mamba**: stacks `n_layers` ResidualBlocks between an embedding and a
  tied LM head (embedding and output projection share weights, halving
  the parameter count).  A greedy `generate()` method is included for
  quick qualitative checks. Note it recomputes the full sequence each step,
  so it is simple but slow.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-8):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        # no mean subtraction - just scale by 1/rms
        # tried layernorm first - mean subtraction hurt training on short sequences
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()
        return x / rms * self.weight


class ResidualBlock(nn.Module):
    def __init__(self, d_model, N, **kwargs):
        super().__init__()
        self.norm = RMSNorm(d_model)
        self.mamba = MambaBlock(d_model, N, **kwargs)

    def forward(self, x):
        # pre-norm residual: output = x + mamba(rmsnorm(x))
        # pre-norm is more stable than post-norm at depth
        return x + self.mamba(self.norm(x))


class Mamba(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, N, **kwargs):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            ResidualBlock(d_model, N, **kwargs) for _ in range(n_layers)
        ])
        self.norm_f = RMSNorm(d_model)
        # lm head tied to embedding weights - halves params, standard practice for lm
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.embedding.weight

    def forward(self, x):
        # x: (seq_len,) token ids
        x = self.embedding(x)     # (seq_len, d_model)
        for layer in self.layers:
            x = layer(x)
        x = self.norm_f(x)
        return self.lm_head(x)    # (seq_len, vocab_size)

    @torch.no_grad()
    def generate(self, prompt_ids, max_new_tokens, temperature=1.0):
        # prompt_ids: list of int token ids
        # no kv cache - recomputes full sequence each step, slow but simple
        tokens = list(prompt_ids)
        for _ in range(max_new_tokens):
            x = torch.tensor(tokens, dtype=torch.long)
            logits = self(x)[-1]   # only need last token's logits
            if temperature != 1.0:
                logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1).item()
            tokens.append(next_token)
        return tokens


# quick sanity check - shapes only, no training
if __name__ == '__main__' or True:
    _vocab = vocab_size  # from cell 11
    _model = Mamba(vocab_size=_vocab, d_model=64, n_layers=2, N=8)
    _ids = torch.tensor(encode(text[:20]), dtype=torch.long)
    _logits = _model(_ids)
    print(f'input tokens:  {_ids.shape}')
    print(f'output logits: {_logits.shape}')  # should be (20, vocab_size)
    _gen = _model.generate(encode(text[:5]), max_new_tokens=10, temperature=0.8)
    print(f'generated: {decode(_gen)}')
    print('Mamba full model OK')


### 4. Selective Discretization Verification

Before relying on `discretize_ssm_zoh_selective` inside `MambaBlock`, this
cell confirms that the diagonal-A shortcut produces the correct output shapes.

The diagonal formulation replaces the O(N³) matrix-exponential augmentation
used in the general ZOH (section 2 of the SSM intro) with element-wise ops:

```
A_bar = exp(A * Δ)
B_bar = ((A_bar - 1) / A) * B_k
```

This is only valid when A is diagonal, but that constraint is exactly what
Mamba enforces, and it makes the per-token selective scan fast enough to
run in practice.

In [ ]:
# Selective SSM (the full Mamba block builds on this)

# Define dimensions
N = 4  # State dimension
D_in = 2 # Input dimension
D_out = 1 # Output dimension

# Continuous-time A matrix (fixed, learned parameter)
A_cont = torch.randn(N, N)

# D matrix (fixed, typically small or zero)
D_cont = torch.randn(D_out, D_in)

# Initial hidden state
h_0 = torch.zeros(N)

# Simulate a sequence (e.g., 5 steps) with selective parameters
sequence_length = 5
inputs = torch.randn(sequence_length, D_in)
hidden_states_selective = [h_0]
outputs_selective = []

current_h = h_0
for i in range(sequence_length):
    x_k = inputs[i]

    # Selective parameters generation (simplified for demonstration)
    # In a real Mamba, these would be generated by MLPs from x_k
    delta_k = torch.sigmoid(torch.randn(1)) # Example: delta_k generated from x_k, then scaled
    B_k_cont = torch.randn(N, D_in) # Example: B_k generated from x_k
    C_k_cont = torch.randn(D_out, N) # Example: C_k generated from x_k

    # Discretize for the current step with selective parameters
    A_bar, B_bar = discretize_ssm_zoh_selective(A_cont, B_k_cont, delta_k)

    # Perform one step of the discrete SSM
    current_h, y_k = discrete_ssm_step_selective(current_h, x_k, A_bar, B_bar, C_k_cont, D_cont)

    hidden_states_selective.append(current_h)
    outputs_selective.append(y_k)

print(f"\nSimulated Selective SSM sequence over {sequence_length} steps:")
for i in range(sequence_length):
    print(f"Step {i+1}: Input={inputs[i]}, Output={outputs_selective[i]}")



Simulated Selective SSM sequence over 5 steps:
Step 1: Input=tensor([1.7775, 0.4740]), Output=tensor([1.7281])
Step 2: Input=tensor([-2.1607,  0.8597]), Output=tensor([5.7177])
Step 3: Input=tensor([-1.6761,  0.7996]), Output=tensor([13.4279])
Step 4: Input=tensor([-0.1125, -0.8878]), Output=tensor([-63.5860])
Step 5: Input=tensor([-0.6231, -0.2251]), Output=tensor([-71.9178])


### 5. Label-Smoothed BCE Loss for Pathfinder

Standard cross-entropy is brittle on binary tasks like Pathfinder because the
model is pushed toward hard 0/1 probabilities, which amplifies overconfidence and
destabilises training on long sequences.  Label smoothing mixes the true target
with a uniform distribution, keeping the model calibrated:

```
loss = (1 - ε) · CE(logits, y)  +  ε · CE(logits, 0.5)
```

For binary classification `CE(logits, 0.5)` is the cross-entropy against the
uniform distribution `[0.5, 0.5]`.  `ε = 0.1` is the default.

In [ ]:
import torch.nn.functional as F


def label_smoothed_bce_loss(logits, labels, eps=0.1):
    """Label-smoothed cross-entropy for Pathfinder / Path-X binary classification.

    Combines the standard (hard) cross-entropy with a soft cross-entropy against
    the uniform distribution over classes:

        loss = (1 - eps) * CE(logits, y) + eps * CE(logits, uniform)

    For binary classification (C=2) the uniform target is [0.5, 0.5], so the
    soft term equals CE(logits, 0.5) as stated in the LRA Mamba fix spec.

    Args:
        logits : (B, num_classes), raw model outputs, not softmaxed.
        labels : (B,) long, ground-truth class indices.
        eps    : float, smoothing weight in [0, 1]; default 0.1.

    Returns:
        Scalar loss tensor.
    """
    # Hard term: standard cross-entropy against ground-truth labels.
    ce_hard = F.cross_entropy(logits, labels)

    # Soft term: cross-entropy against the uniform distribution over C classes.
    # For C=2 this equals -0.5*log(p0) - 0.5*log(p1) = CE(logits, 0.5).
    log_probs = F.log_softmax(logits, dim=-1)        # (B, C)
    ce_soft   = -log_probs.mean(dim=-1).mean()       # mean over classes then batch

    return (1.0 - eps) * ce_hard + eps * ce_soft

### 6. Δ-Projection Weight Decay

The delta projection (`delta_proj`) controls the step size Δ that gates how
fast the hidden state decays.  Applying a stronger L2 penalty specifically to
its weights regularises them toward smaller magnitudes, which combined with
the negative bias init from earlier biases the model toward small Δ values
and therefore longer memory retention throughout training.

`get_optimizer` splits model parameters into two groups:
- **default group**: standard weight decay (e.g. 0.01)
- **delta_proj group**: stronger weight decay `dt_proj_wd` (default `1e-3`)

Both groups share the same learning rate; only the decay differs.

In [ ]:
import torch


def get_optimizer(
    model,
    lr=1e-3,
    base_wd=0.01,
    dt_proj_wd=1e-3,
):
    """AdamW optimizer with a dedicated parameter group for delta_proj.

     delta projection weights receive a separate, configurable weight decay
    (dt_proj_wd) that regularises them toward small magnitudes.  Smaller weights
    mean smaller raw pre-activations, which softplus maps to smaller Δ values,
    biasing the model toward slow state decay and long-range memory retention.

 
    model: nn.Module containing one or more MambaBlock instances.
    lr: Learning rate shared by all parameter groups.
    base_wd: Weight decay for all parameters except delta_proj weights.
    dt_proj_wd: Weight decay applied only to delta_proj weight matrices.
                     Defaults to 1e-3. Set to 0 to disable.

    Returns torch.optim.AdamW configured with two parameter groups.
    """
    dt_proj_weights = []
    other_params    = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        # Target only the weight matrix of every delta_proj linear layer;
        # biases are excluded so the negative-bias init is not penalised.
        if 'delta_proj.weight' in name:
            dt_proj_weights.append(param)
        else:
            other_params.append(param)

    param_groups = [
        {'params': other_params,    'weight_decay': base_wd,    'name': 'default'},
        {'params': dt_proj_weights, 'weight_decay': dt_proj_wd, 'name': 'delta_proj'},
    ]

    return torch.optim.AdamW(param_groups, lr=lr)

### 7. Proof: Does It Help? Baseline vs. Improved

Train two identical Mamba models on the Romeo & Juliet text:
- **Baseline**: standard cross-entropy + plain AdamW (no label smoothing, no delta-proj weight decay)
- **Improved**: label-smoothed CE (ε=0.1) + delta-proj weight decay optimizer from sections 5 & 6

Same architecture, same random seed, same number of steps. Lower val loss = better.

In [ ]:
import torch, torch.nn.functional as F

torch.manual_seed(42)

STEPS      = 500
LR         = 3e-3
D_MODEL    = 32
N_LAYERS   = 2
N_STATE    = 8
BLOCK_SIZE = 16
BATCH_SIZE = 8

def train(use_label_smooth, use_dt_wd, tag):
    torch.manual_seed(42)
    model = Mamba(vocab_size=vocab_size, d_model=D_MODEL, n_layers=N_LAYERS, N=N_STATE)

    if use_dt_wd:
        opt = get_optimizer(model, lr=LR, base_wd=0.01, dt_proj_wd=1e-3)
    else:
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

    for step in range(STEPS):
        x, y = get_batch('train', BLOCK_SIZE, BATCH_SIZE)
        # The model processes one sequence at a time (no batch dimension in MambaBlock).
        # Stack results back into (B, T, vocab) for the loss.
        logits = torch.stack([model(x[b]) for b in range(x.shape[0])])
        B, T, V = logits.shape
        flat_logits = logits.reshape(B * T, V)
        flat_y      = y.reshape(B * T)

        if use_label_smooth:
            loss = label_smoothed_bce_loss(flat_logits, flat_y, eps=0.1)
        else:
            loss = F.cross_entropy(flat_logits, flat_y)

        opt.zero_grad()
        loss.backward()
        opt.step()

    model.eval()
    with torch.no_grad():
        xv, yv = get_batch('val', BLOCK_SIZE, BATCH_SIZE)
        vlogits = torch.stack([model(xv[b]) for b in range(xv.shape[0])]).reshape(-1, vocab_size)
        val_loss = F.cross_entropy(vlogits, yv.reshape(-1)).item()
    print(f"{tag:30s}  val loss = {val_loss:.4f}")
    return val_loss

base = train(use_label_smooth=False, use_dt_wd=False, tag="Baseline")
impr = train(use_label_smooth=True,  use_dt_wd=True,  tag="Label smooth + delta WD")
print(f"\nImprovement: {(base - impr):.4f} nats  ({100*(base-impr)/base:.1f}% lower val loss)")


### 8. Unit Tests

Each test below checks one function or module introduced in this notebook.
The tests are intentionally short: they verify shapes, types, and key
numerical properties rather than training a full model.

Running this cell should print a series of PASSED lines with no errors.

In [ ]:
import torch
import torch.nn.functional as F
import math

def check(name, condition):
    """Print PASSED or raise an AssertionError with the test name."""
    assert condition, f"FAILED: {name}"
    print(f"PASSED: {name}")


# Test 1: ZOH discretization produces the right output shapes.
# A_bar should match A in size; B_bar should match B in size.
N, D_in = 4, 2
A = torch.randn(N, N)
B = torch.randn(N, D_in)
delta = torch.tensor(0.1)
A_bar, B_bar = discretize_ssm_zoh(A, B, delta)
check("discretize_ssm_zoh output shapes", A_bar.shape == (N, N) and B_bar.shape == (N, D_in))


# Test 2: Selective ZOH discretization works with a diagonal A vector.
# The diagonal shortcut avoids the matrix exponential and must give the same shapes.
A_diag = torch.randn(N)
B_k = torch.randn(N, D_in)
delta_k = torch.tensor(0.05)
A_bar_s, B_bar_s = discretize_ssm_zoh_selective(A_diag, B_k, delta_k)
check("discretize_ssm_zoh_selective output shapes",
      A_bar_s.shape == (N,) and B_bar_s.shape == (N, D_in))


# Test 3: MambaBlock preserves the input shape.
# The block should output a tensor of exactly the same shape as its input.
d_model, seq_len = 32, 10
x = torch.randn(seq_len, d_model)
block = MambaBlock(d_model=d_model, N=8)
out = block(x)
check("MambaBlock output shape matches input shape", out.shape == x.shape)


# Test 4: The full Mamba model produces one logit vector per input token.
# Output should be (seq_len, vocab_size).
model = Mamba(vocab_size=vocab_size, d_model=32, n_layers=2, N=8)
ids = torch.tensor(encode(text[:20]), dtype=torch.long)
logits = model(ids)
check("Mamba forward output shape", logits.shape == (len(ids), vocab_size))


# Test 5: generate() returns exactly the right number of tokens.
# The output length should equal the prompt length plus max_new_tokens.
prompt = encode(text[:5])
max_new = 10
generated = model.generate(prompt, max_new_tokens=max_new)
check("Mamba generate length", len(generated) == len(prompt) + max_new)


# Test 6: Label smoothing changes the loss value.
# If smoothing has no effect the two losses would be identical, which would
# indicate a bug. They must differ by at least a small amount.
logits_test = torch.randn(8, 2)
labels_test = torch.randint(0, 2, (8,))
loss_hard = F.cross_entropy(logits_test, labels_test)
loss_smooth = label_smoothed_bce_loss(logits_test, labels_test, eps=0.1)
check("label_smoothed_bce_loss differs from hard CE", abs(loss_smooth.item() - loss_hard.item()) > 1e-6)


# Test 7: Smoothed loss lies between hard CE and uniform CE.
# Mixing in the uniform distribution should pull the loss toward the uniform
# baseline, so the smoothed value must be strictly between the two extremes.
log_probs = F.log_softmax(logits_test, dim=-1)
loss_uniform = -log_probs.mean()
lo = min(loss_hard.item(), loss_uniform.item())
hi = max(loss_hard.item(), loss_uniform.item())
check("smoothed loss is between hard CE and uniform CE",
      lo <= loss_smooth.item() <= hi + 1e-6)


# Test 8: get_optimizer creates exactly two parameter groups.
# One group is for regular parameters and one is for delta_proj weights only.
small_block = MambaBlock(d_model=32, N=8)
opt = get_optimizer(small_block, lr=1e-3, base_wd=0.01, dt_proj_wd=1e-3)
check("get_optimizer creates two parameter groups", len(opt.param_groups) == 2)


# Test 9: The delta_proj group has its own separate weight decay value.
# This value must match the dt_proj_wd argument passed to get_optimizer.
dt_wd = opt.param_groups[1]['weight_decay']
check("delta_proj group weight decay is correct", abs(dt_wd - 1e-3) < 1e-9)


# Test 10: The delta_proj parameter group is non-empty.
# If no parameters matched the filter the group would be useless.
n_dt_params = sum(p.numel() for p in opt.param_groups[1]['params'])
check("delta_proj group contains parameters", n_dt_params > 0)


print("\nAll tests passed.")
